# EE/CS 148B HW2: Colab Setup

This notebook is meant to be imported directly into Colab to provide a quickstart environment setup.


## Colab Setup

Before running:

- Switch the runtime to a GPU runtime.
- We recommend using an A100 runtime for the GRPO experiments.
- Put the `hw2` directory somewhere accessible from Colab, typically Google Drive.

This notebook assumes the repo already exists and only sets up Python dependencies plus the repo import path.

Note: We will be using `pip` for dependencies inside Colab.

In [ ]:
%%capture
!pip -q install -U vllm pytest datasets transformers accelerate sentencepiece matplotlib pandas tqdm sympy pylatexenc latex2sympy2_extended "math-verify[antlr4-13-2]"

`vllm` is installed and used by default for the direct, CoT, and self-consistency GSM8K baselines. If vLLM install or initialization fails in Colab, set `USE_VLLM = False` in the config cell to fall back to HuggingFace generation.


In [ ]:
from pathlib import Path

USE_DRIVE = True
DRIVE_REPO_ROOT = Path('/content/drive/MyDrive/hw2/')  # edit if needed
LOCAL_REPO_ROOT = Path('/content/hw2-sols')

if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    REPO_ROOT = DRIVE_REPO_ROOT
else:
    REPO_ROOT = LOCAL_REPO_ROOT

assert REPO_ROOT.exists(), f'Repo root does not exist: {REPO_ROOT}'
print('Using repo:', REPO_ROOT)


In [ ]:
import os
import sys

sys.path.insert(0, str(REPO_ROOT))
sys.path.insert(0, str(REPO_ROOT / 'basics'))
os.chdir(REPO_ROOT)
print('cwd =', os.getcwd())

In [ ]:
import gc
import json
import random
import subprocess
from collections import Counter
from pathlib import Path
from types import SimpleNamespace

import matplotlib.pyplot as plt
import pandas as pd
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, set_seed

from alignment.eval import load_gsm8k_examples, build_prompts, evaluate_vllm
from alignment.prompts import DIRECT_PROMPT_TEMPLATE, COT_PROMPT_TEMPLATE
from alignment.rewards import answer_tag_reward_fn
from alignment.grpo import train_grpo

SEED = 0
set_seed(SEED)
random.seed(SEED)

if torch.cuda.is_available():
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.set_float32_matmul_precision('high')

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device =', DEVICE)
if torch.cuda.is_available():
    print('gpu =', torch.cuda.get_device_name(0))
    try:
        print(subprocess.run(['nvidia-smi', '-L'], capture_output=True, text=True, check=False).stdout)
    except FileNotFoundError:
        pass


# Your code starts here!